In [1]:
import pandas as pd
import multiprocessing as mp
from ete3 import Tree
import pickle
from ete3 import NCBITaxa, Tree

/n/home11/rkapoor/.conda/envs/rishabh/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import pickle
with open('outputs/connected_components.pickle', 'rb') as f:
    connected_components = pickle.load(f)
with open('outputs/connected_components_reps.pickle', 'rb') as f:
    representative = pickle.load(f)

In [3]:

def init_worker():
    global ncbi
    ncbi = NCBITaxa()
ncbi = NCBITaxa()


def get_ncbi_lineage_by_rank(species_name, prefer_rank="species"):
    
    """
    Return NCBI taxid and full lineage as {rank: scientific_name}
    for a species name.
    """
    global ncbi
    try:
        if ncbi is None:
            # Fallback for non-multiprocessing use
            ncbi = NCBITaxa()
        species_name = species_name.strip()
    
        if not species_name:
            raise ValueError("species_name is empty")
        name2taxids = ncbi.get_name_translator([species_name])
    
    
        if species_name not in name2taxids:
            if len(species_name.split(" "))>2:
                species_name=" ".join(species_name.split(" ")[0:2])
                get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                try: 
                    return get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                except:
                    print(species_name)
                    return 'None','None'
            raise ValueError(f"No NCBI taxid found for: {species_name!r}")
    
        candidate_taxids = name2taxids[species_name]
    
        if len(candidate_taxids) > 1:
            candidate_ranks = ncbi.get_rank(candidate_taxids)
            preferred = [
                taxid for taxid in candidate_taxids
                if candidate_ranks.get(taxid) == prefer_rank
            ]
            taxid = preferred[0] if preferred else candidate_taxids[0]
        else:
            taxid = candidate_taxids[0]
    
        lineage_taxids = ncbi.get_lineage(taxid)
        lineage_names = ncbi.get_taxid_translator(lineage_taxids)
        lineage_ranks = ncbi.get_rank(lineage_taxids)
    
        lineage_by_rank = {
            lineage_ranks[t]: lineage_names[t]
            for t in lineage_taxids
            if lineage_ranks.get(t) not in (None, "no rank")
        }
        return taxid, lineage_by_rank
    except:
        return 'None',{'None':'None'}
    

In [4]:

alldf=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)
dipt_proteins=set(alldf[alldf.slineage.astype(str).str.contains('Diptera')]['accession'])
htgdf=alldf[alldf.HGT]
htgdf=htgdf.set_index('accession')
htgs=set(htgdf.index)

import pickle
with open('outputs/connected_components.pickle', 'rb') as f:
    connected_components = pickle.load(f)
with open('outputs/connected_components_reps.pickle', 'rb') as f:
    representative = pickle.load(f)

In [5]:
def get_sister_info(ranks):
    domain = kingdom = phylum = classt = order = None
    try:
        if 'Viruses' in str(ranks):
            domain='Viruses'
        else:
            domain=ranks['domain']
    except:
        superkingdom='None'
    try:
        kingdom=ranks['kingdom']
    except:
        kingdom='None'
    try:
        phylum=ranks['phylum']
    except:
        phylum='None'
    try:
        classt=ranks['class']
    except:
        classt='None'
    try:
        order=ranks['order']
    except:
        order='None'
    return [domain, kingdom, phylum, classt, order]


In [6]:
def get_sister_cousin(node):

    parent = node.up
    grandparent = parent.up if parent is not None else None
    
    # Sister clades: children of parent that are not the focal node
    sisters = []
    if parent is not None:
        for child in parent.children:
            if child is not node:
                sisters.extend(child.iter_leaves())
    
    # Cousin clades: children of grandparent that are not the parent clade
    cousins = []
    if grandparent is not None:
        for child in grandparent.children:
            if child is not parent:
                cousins.extend(child.iter_leaves())
    return sisters, cousins

In [2]:
import os
tdo=[x for x in os.listdir("outputs/phylogenetic_dataset") if "rev_aa.contree" not in os.listdir(f"outputs/phylogenetic_dataset/{x}")]  

In [8]:
tdo.sort()

dfresult=pd.DataFrame()

In [9]:
def get_data(i):
    try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")
        
        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            l = pickle.load(f)
        
        tdn = list(l.keys())
        td = [
            x.split("; ")[-1].split(" x ")[0]
            if not x.split("; ")[-1].split(" x ")[0].isdigit()
            else x.split("; ")[-2].split(" x ")[0]
            for x in l.values()
        ]
        with mp.Pool(processes=50, initializer=init_worker) as pool:
            results = pool.map(get_ncbi_lineage_by_rank, td)
        lmap = dict(zip(tdn, [x[1] for x in results]))
        tmap= dict(zip(tdn, [x[0] for x in results]))
            
        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
                leaf.add_features(metazoa= "Metazoa" in str(l) and 'Sciaridae' not in str(l))
                leaf.add_features(diptera= "Sciaridae" not in str(l) and 'Diptera' in str(l))
                leaf.add_features(taxid= tmap[leaf.name])
                
                if 'species' in l:
                    leaf.add_features(species=l['species'])
        
                if 'order' in l and l['order']=='Diptera':
                    if 'family' in l and l['family']=="Sciaridae":
                        leaf.add_features(tax_label= "Sciaridae")
                    else:
                        
                        leaf.add_features(tax_label= "Diptera")
                elif 'class' in l and l['class']=='Insecta' :
                    leaf.add_features(tax_label=l['order'])
                elif 'phylum' in l and l['phylum']=='Arthropoda' :
                    leaf.add_features(tax_label=l['class'])
                elif 'kingdom' in l and l['kingdom']=='Metazoa':
                    leaf.add_features(tax_label=l['phylum'])
                elif 'Viruses' in str(l):
                    leaf.add_features(tax_label='Viruses')
                elif 'Fungi' in str(l):
                     leaf.add_features(tax_label='Fungi')
                elif 'Viridiplantae' in str(l):
                     leaf.add_features(tax_label='Viridiplantae')
                elif 'domain' in l and 'Eukaryota'==l['domain']:
                    leaf.add_features(tax_label='Protist')
                elif 'domain' in l and l['domain']=='Bacteria':
                    leaf.add_features(tax_label='Bacteria')
                elif 'domain' in l and l['domain']=='Archaea':
                    leaf.add_features(tax_label='Archaea')
                else:
                    leaf.add_features(tax_label='None')
        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
        
            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
        for node in mono:
            accept = False
            node_og=node
            while node.up:
                
                check=len([x for x in t if x.tax_label=='Diptera'])>0
                if not check:
                    accepted_nodes.append(node_og)
                    break
                if node.up.support>70 and 'Diptera' not in set([x.tax_label for x in node.up]):
                    accepted_nodes.append(node_og)
        
                    break
                elif 'Diptera' in set([x.tax_label for x in node.up]):
                    rejected_nodes.append(node_og)
                    break
                node=node.up
        non_metazoan_transfer=False  
        
        ni=1
        done=[]
        for node in accepted_nodes:
            if len(set([x.name for x in node])&htgs)>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Accept'
                dfresult.loc[f"{i}.{ni}","n_cluster_seqs"]=len(set([x.name for x in node])&set(connected_components[i]))
                sciarid_species=list(set([x.species for x in node]))
                sciarid_species.sort()
                dfresult.loc[f"{i}.{ni}","sciarid_species"]=str(sciarid_species)
        
                sisters,cousins=get_sister_cousin(node)
        
                sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}
                cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}
    
                if 'Sciaridae' in sister_taxlabel or 'Sciaridae' in cousin_taxlabel:
                    dfresult.loc[f"{i}.{ni}","sciarid_sister"]=True
                dfresult.loc[f"{i}.{ni}","sister_tax"]=str(sister_taxlabel-{'Sciaridae'})
                dfresult.loc[f"{i}.{ni}","cousin_tax"]=str(cousin_taxlabel-{'Sciaridae'})
            
                
                
        
                sister_tids=[x.taxid for x in sisters if x.tax_label!='Sciaridae']
             
                top=ncbi.get_topology(sister_tids)
                name=ncbi.get_taxid_translator([top.taxid])[top.taxid]
                dfresult.loc[f"{i}.{ni}","sister_name"]=str(name)
                l=ncbi.get_lineage(top.taxid)
            
                ranks={ncbi.get_rank([x])[x]:ncbi.get_taxid_translator([x])[x] for x in l}
                r=get_sister_info(ranks)
                dfresult.loc[f"{i}.{ni}",['sister_domain','sister_kingdom','sister_phylum','sister_class','sister_order']]=r
               
                non_metazoa={'Protist','Bacteria','Viridiplantae','Fungi', 'Viruses','Fungi','Archaea'}
                if len(cousin_taxlabel&non_metazoa)>0 or len(sister_taxlabel&non_metazoa)>0:
                    non_metazoan_transfer=True
                done.append(f"{i}.{ni}")
        
                paralogs=set([x.name for x in node if x.name in set(htgs)])
                if len(rejected_nodes)>0:
                    dfresult.loc[f"{i}.{ni}","pseudoparalog"]=True
                if len(paralogs)>1:
                     dfresult.loc[f"{i}.{ni}","post_transfer_duplication"]=True
                     dfresult.loc[f"{i}.{ni}","n_paralogs"]=len(paralogs)
                if len(accepted_nodes)>0:
                     dfresult.loc[f"{i}.{ni}","multi_transfer"]=True
                ni+=1
        for node in rejected_nodes:
            if len([ x for x in node if 'XP_' in x.name])>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Reject'
                ni+=1
        if non_metazoan_transfer:
            mono=[]
            for node in t.get_monophyletic(values=[True], target_attr="metazoa"):
                mono.append(node)
            convergent_transfers=set()
            for y in mono:
                if len(set([x.tax_label for x in y]))==1:
                    sisters,cousins=get_sister_cousin(y)
                    sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}-{'Sciaridae'}
                    cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}-{'Sciaridae'}
                    if len(sister_taxlabel&non_metazoa)>0 and len(cousin_taxlabel&non_metazoa)>0:
                        convergent_transfers=convergent_transfers|set([x.tax_label for x in y])
            dfresult.loc[done,'convergent']=str(convergent_transfers)
    except:
        print('fail',i)

In [10]:
for x in tdo:
 
    get_data(int(x))

In [12]:
len(connected_components)

924

In [18]:
from collections import Counter
Counter(dfresult[dfresult.result=='Accept'].sister_kingdom)

Counter({'Metazoa': 1007,
         'Viridiplantae': 17,
         'Pseudomonadati': 50,
         'Bacillati': 23,
         'None': 122,
         'Fungi': 72,
         'Thermoproteati': 11,
         'Thermotogati': 1,
         'Bamfordvirae': 1})

In [13]:
dfresult.to_csv("outputs/inspection_table2.tsv",sep='\t')

In [15]:
dfresult.to_csv('outputs/manual_inspection_table.tsv',sep='\t')

In [100]:


def get_data(i):
    print(i)
    try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")
        
        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            l = pickle.load(f)
        
        tdn = list(l.keys())
        td = [
            x.split("; ")[-1].split(" x ")[0]
            if not x.split("; ")[-1].split(" x ")[0].isdigit()
            else x.split("; ")[-2].split(" x ")[0]
            for x in l.values()
        ]
        with mp.Pool(processes=50, initializer=init_worker) as pool:
            results = pool.map(get_ncbi_lineage_by_rank, td)
        lmap = dict(zip(tdn, [x[1] for x in results]))
        tmap= dict(zip(tdn, [x[0] for x in results]))
            
        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
                leaf.add_features(metazoa= "Metazoa" in str(l) and 'Sciaridae' not in str(l))
                leaf.add_features(diptera= "Sciaridae" not in str(l) and 'Diptera' in str(l))
                leaf.add_features(taxid= tmap[leaf.name])
                if 'species' in l:
                    leaf.add_features(species=l['species'])
                if 'order' in set(l.keys()) and l['order']=='Diptera':
                    if 'family' in l and l['family']=="Sciaridae":
                        leaf.add_features(tax_label= "Sciaridae")
                    else:
                        
                        leaf.add_features(tax_label= "Diptera")
                elif 'class' in l and l['class']=='Insecta' :
                    leaf.add_features(tax_label=l['order'])
                elif 'phylum' in l and l['phylum']=='Arthropoda' :
                    leaf.add_features(tax_label=l['class'])
                elif 'kingdom' in l and l['kingdom']=='Metazoa':
                    leaf.add_features(tax_label=l['phylum'])
                elif 'Viruses' in str(l):
                    leaf.add_features(tax_label='Viruses')
                elif 'Fungi' in str(l):
                     leaf.add_features(tax_label='Fungi')
                elif 'Viridiplantae' in str(l):
                     leaf.add_features(tax_label='Viridiplantae')
                elif 'Eukaryota'==l['domain']:
                    leaf.add_features(tax_label='Protist')
                elif l['domain']=='Bacteria':
                    leaf.add_features(tax_label='Bacteria')
                elif l['domain']=='Archaea':
                    leaf.add_features(tax_label='Archaea')
                
        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
        for node in mono:
            accept = False
            # print(node_og)
            while node.up:
                
                check=len([x for x in t if x.tax_label=='Diptera'])>0
                if not check:
                    accepted_nodes.append(node)
                    break
                if node.up.support>70 and 'Diptera' not in set([x.tax_label for x in node.up]):
                    accepted_nodes.append(node)
                   
                    break
                elif 'Diptera' in set([x.tax_label for x in node.up]):
                    rejected_nodes.append(node)
                    break
                node=node.up
        non_metazoan_transfer=False  
        
        ni=1
        done=[]
        for node in accepted_nodes:
            if len([ x for x in node if 'XP_' in x.name])>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Accept'
                dfresult.loc[f"{i}.{ni}","n_cluster_seqs"]=len(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","sciarid_species"]=len(set([x.species for x in node]))
                
                sisters,cousins=get_sister_cousin(node)
                sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}-{'Sciaridae'}
                cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}-{'Sciaridae'}
                dfresult.loc[f"{i}.{ni}","sister_tax"]=str(sister_taxlabel)
                dfresult.loc[f"{i}.{ni}","cousin_tax"]=str(cousin_taxlabel)
            
                
                
        
                sister_tids=[x.taxid for x in sisters if x.tax_label!='Sciaridae']
                top=ncbi.get_topology(sister_tids)
                name=ncbi.get_taxid_translator([top.taxid])[top.taxid]
                dfresult.loc[f"{i}.{ni}","sister_name"]=str(name)
                l=ncbi.get_lineage(top.taxid)
            
                ranks={ncbi.get_rank([x])[x]:ncbi.get_taxid_translator([x])[x] for x in l}
                r=get_sister_info(ranks)
                dfresult.loc[f"{i}.{ni}",['sister_domain','sister_kingdom','sister_phylum','sister_class','sister_order']]=r
               
                non_metazoa={'Protist','Bacteria','Viridiplantae','Fungi', 'Viruses','Fungi','Archaea'}
                if len(cousin_taxlabel&non_metazoa)>0 or len(sister_taxlabel&non_metazoa)>0:
                    non_metazoan_transfer=True
                done.append(f"{i}.{ni}")
    
                paralogs=set([x.name for x in node if x.name in set(htgs)])
                if len(rejected_nodes)>0:
                    dfresult.loc[f"{i}.{ni}","pseudoparalog"]=True
                if len(paralogs)>0:
                     dfresult.loc[f"{i}.{ni}","post_transfer_duplication"]=True
                     dfresult.loc[f"{i}.{ni}","n_paralogs"]=len(paralogs)
                if len(accepted_nodes)>0:
                     dfresult.loc[f"{i}.{ni}","multi_transfer"]=True
                ni+=1
        for node in rejected_nodes:
            if len([ x for x in node if 'XP_' in x.name])>0:
                dfresult.loc[f"{i}.{ni}","all_seqs"]=str([x.name for x in node])
                dfresult.loc[f"{i}.{ni}","cluster_seqs"]=str(set([x.name for x in node])&set(connected_components[i]))
                dfresult.loc[f"{i}.{ni}","result"]='Reject'
                ni+=1
        if non_metazoan_transfer:
            mono=[]
            for node in t.get_monophyletic(values=[True], target_attr="metazoa"):
                mono.append(node)
            convergent_transfers=set()
            for y in mono:
                if len(set([x.tax_label for x in y]))==1:
                    sisters,cousins=get_sister_cousin(y)
                    sister_taxlabel = {getattr(x, "tax_label", None) for x in sisters}-{'Sciaridae'}
                    cousin_taxlabel = {getattr(x, "tax_label", None) for x in cousins}-{'Sciaridae'}
                    if len(sister_taxlabel&non_metazoa)>0 and len(cousin_taxlabel&non_metazoa)>0:
                        convergent_transfers=convergent_transfers|set([x.tax_label for x in y])
            dfresult.loc[done,'convergent']=str(convergent_transfers)
    except:
        print("fail",i)

    

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
20
21
22
23
24
25
26
fail 26
27
fail 27
28
29
30
31
32
33
34
35
36
37
38
39
40
40
fail 40
41
42
42
43
44
45
46
47
48
49
50
51
52
53
53
54
55
56
fail 56
57
58
59
60
61
62
63
64
64
65
66
67
68
69
70
71
72
73
74
75
76
77
77
78
79
80
81
81
82
83
84
85
86
87
88
89
90
91
92
93
fail 93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
111
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
131
132
133
134
134
135
136
fail 136
137
138
139
140
141
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
157
158
159
160
161
162
163
164
164
165
166
167
168
169
170
171
172
173
173
174
175
176
177
178
179
fail 179
180
181
fail 181
182
183
184
185
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
fail 239
240
241
242
243
244
245
246
2

In [102]:
dfresult[dfresult.result=='Accept'].n_cluster_seqs.sum()


1580.0

In [103]:
dfresult

,all_seqs,cluster_seqs,result,n_cluster_seqs,sciarid_species,sister_tax,cousin_tax,sister_name,sister_domain,sister_kingdom,sister_phylum,sister_class,sister_order,pseudoparalog,multi_transfer,post_transfer_duplication,n_paralogs,convergent
0.1,['XP_037032481.1'],set(),Accept,0.0,1.0,"{'Hemiptera', 'Hymenoptera', 'Ephemeroptera', ...",{'Hymenoptera'},Pterygota,Eukaryota,Metazoa,Arthropoda,Insecta,None,True,True,NaN,NaN,{'Malacostraca'}
0.2,['XP_037029402.1'],set(),Accept,0.0,1.0,"{'Hemiptera', 'Hymenoptera', 'Ephemeroptera', ...",{'Ephemeroptera'},Pterygota,Eukaryota,Metazoa,Arthropoda,Insecta,None,True,True,True,1.0,{'Malacostraca'}
0.3,"['XP_037037108.1', 'XP_037037104.1']",set(),Accept,0.0,1.0,{'Blattodea'},{'Diptera'},Zootermopsis nevadensis,Eukaryota,Metazoa,Arthropoda,Insecta,Blattodea,True,True,NaN,NaN,{'Malacostraca'}
0.4,['XP_037042628.1'],set(),Accept,0.0,1.0,{'Viridiplantae'},"{'Diptera', 'Hemiptera', 'Lepidoptera', 'Neuro...",Olea europaea var. sylvestris,Eukaryota,Viridiplantae,Streptophyta,Magnoliopsida,Lamiales,True,True,True,1.0,{'Malacostraca'}
0.5,['XP_037029759.1'],set(),Accept,0.0,1.0,{'Viridiplantae'},set(),Olea europaea var. sylvestris,Eukaryota,Viridiplantae,Streptophyta,Magnoliopsida,Lamiales,True,True,True,1.0,{'Malacostraca'}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
917.1,['XP_080120532.1'],{'XP_080120532.1'},Accept,1.0,1.0,{'Nematoda'},{'Nematoda'},Caenorhabditis nigoni,Eukaryota,Metazoa,Nematoda,Chromadorea,Rhabditida,NaN,True,True,1.0,NaN
919.1,"['XP_080120575.1', 'bimpatiensg28229.t1', 'bim...",{'XP_080120575.1'},Accept,1.0,2.0,"{'Archaea', 'Bacteria'}",{'Bacteria'},cellular organisms,None,None,None,None,None,NaN,True,True,1.0,set()
920.1,"['bimpatiensg21749.t1', 'XP_080120592.1', 'XP_...","{'XP_080120592.1', 'XP_080120593.1'}",Accept,2.0,2.0,{'Collembola'},set(),Allacma fusca,Eukaryota,Metazoa,Arthropoda,Collembola,Symphypleona,NaN,True,True,2.0,NaN
921.1,"['XP_080120691.1', 'bimpatiensg30905.t1', 'lin...",{'XP_080120691.1'},Accept,1.0,4.0,{'Collembola'},{'Collembola'},Orchesella,Eukaryota,Metazoa,Arthropoda,Collembola,Entomobryomorpha,NaN,True,True,1.0,NaN


In [57]:
old="""XP_037041147.1
XP_037051528.1
XP_037047822.1
XP_037042473.1
XP_037036837.1
XP_037042500.1
XP_037050347.1
XP_037042473.1
XP_037049870.1
XP_037047821.1"""

In [58]:
for x in set(old.split("\n")):
    print(x,[y for y in connected_components if x in connected_components[y]])

XP_037051528.1 [723]
XP_037042500.1 [523]
XP_037036837.1 [350]
XP_037047822.1 []
XP_037047821.1 [648]
XP_037049870.1 [691]
XP_037050347.1 [704]
XP_037042473.1 [520]
XP_037041147.1 [485]


In [54]:
htgdf[htgdf.gene=='LOC119082397']

,n_non_diptera,n_out_fam_diptera,AI,sseqid,slineage,sciname,tid,evalue,bitscore,qcovhsp,qstart,qend,pident,HGT,type,gene,length,description
accession,,,,,,,,,,,,,,,,,,
XP_037047821.1,1998.0,0.0,NaN,KAI1293867.1,cellular organisms; Eukaryota; Opisthokonta; M...,Halotydeus destructor,2874060.0,3.110000e-125,369.0,97.2,7.0,286.0,64.3,True,non_dipt,LOC119082397,288,nicotinate-nucleotide pyrophosphorylase [carbo...


In [43]:

[x for x in connected_components if 'XP_037042473.1' in connected_components[x]]

[520]

In [86]:
ncbi.get_taxid_translator(top.taxid)

TypeError: 'int' object is not iterable

In [23]:
with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
    l = pickle.load(f)
[x for x in l if '3696986' in l[x]]

['GAC2617167.1']

In [24]:
l['GAC2617167.1']

'cellular organisms; Bacteria; Pseudomonadati; Pseudomonadota; Betaproteobacteria; Nitrosomonadales; Nitrosomonadaceae; Nitrosomonas; unclassified Nitrosomonas; 3696986'

In [16]:
mono=[]
for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
    mono.append(node)
    print(node)


   /-XP_037035339.1
--|
   \-XP_037038331.1

--XP_080118957.1

   /-XP_037047824.1
--|
   \-XP_037051163.1

         /-KAG4074008.1
      /-|
   /-|   \-XP_037033438.1
  |  |
--|   \-lingenuag4173.t1
  |
   \-bimpatiensg14011.t1
